[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab07.ipynb)

# Bab 7 — Regresi: Lingkar Kepala Janin (HC18) dan Derajat Retinopati Ordinal (RetinaMNIST)
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4**, lalu **Runtime → Run all**.

Isi:
1. **HC18** (van den Heuvel dkk., 2018; CC BY 4.0): 999 citra USG berlabel lingkar kepala (HC, mm) dan ukuran piksel (mm/piksel).
   Pembagian 70/10/20 **per subjek** (berkas `014_HC`, `014_2HC`, … dianggap satu subjek).
2. Regresi langsung HC dengan ResNet-18 (pra-latih vs acak, 3 seed), target dinormalisasi; MAE/RMSE dalam **mm**, CI 95 % bootstrap,
   plot prediksi vs acuan dan **Bland–Altman** pendahuluan.
3. **RetinaMNIST 128 px** (MedMNIST v2, CC BY 4.0; sumber DeepDRiD): derajat retinopati 0–4 sebagai masalah **ordinal** —
   klasifikasi biasa (CE) vs regresi (MSE + pembulatan) vs ordinal kumulatif (gaya CORAL); metrik **quadratic weighted kappa (QWK)**.

Hasil → `/content/hasil_bab07/` → `/content/hasil_bab07.zip`. Model hanya untuk pembelajaran, **bukan** untuk penggunaan klinis.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 7.0 Lingkungan

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB07_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB07_OUT", "/content/hasil_bab07"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

In [ ]:
with catat_waktu("1_pip_install"):
    jalankan('pip install -q "medmnist==3.0.2"')
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib, matplotlib.ticker; import matplotlib.pyplot as plt
import torchvision, sklearn, medmnist
from medmnist import INFO
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
         "numpy": np.__version__, "scikit-learn": sklearn.__version__, "medmnist": medmnist.__version__,
         "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))
def npz_medmnist(flag, size):
    fn = berkas_medmnist(flag, size); folder = pastikan_dataset("medmnist", [fn]); return np.load(folder / fn)
SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()
# ---------- Perangkat latih bersama (sama di notebook bab 04 dst.) ----------
MEAN_IN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); SD_IN = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
AMP = DEV.type == "cuda"

def ke_tensor_uint8(X):
    """(N,H,W) atau (N,H,W,C) uint8 -> tensor (N,C,H,W) uint8; disimpan di GPU bila ada (data kecil)."""
    t = torch.as_tensor(X)
    t = t[:, None] if t.ndim == 3 else t.permute(0, 3, 1, 2)
    return t.contiguous().to(DEV)

def augmentasi_gpu(x, flip_h=False, flip_v=False, rot90=False, rot_derajat=0.0, geser=0.0, kecerahan=0.1, kontras=0.1, warna=0.0):
    """x float (N,C,H,W) di [0,1]. Semua acak per sampel kecuali rot90 (per batch)."""
    n = x.shape[0]; d = x.device
    if flip_h: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(3), x)
    if flip_v: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(2), x)
    if rot90: x = torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), (2, 3))
    if rot_derajat or geser:
        a = (torch.rand(n, device=d) * 2 - 1) * rot_derajat * np.pi / 180
        t = (torch.rand(n, 2, device=d) * 2 - 1) * geser * 2
        th = torch.stack([torch.stack([a.cos(), -a.sin(), t[:, 0]], 1), torch.stack([a.sin(), a.cos(), t[:, 1]], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode="border", align_corners=False)
    if warna and x.shape[1] == 3: x = x * (1 + (torch.rand(n, 3, 1, 1, device=d) * 2 - 1) * warna)
    m = x.mean((1, 2, 3), keepdim=True)
    x = ((x - m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kontras) + m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kecerahan)
    return x.clamp(0, 1)

class ResNet18(nn.Module):
    """ResNet-18 torchvision; masukan 1 atau 3 kanal [0,1], opsional diperbesar ke `uk` piksel, normalisasi ImageNet."""
    def __init__(self, k, pretrain=True, uk=None, p_drop=0.0):
        super().__init__(); self.uk = uk
        w = None
        if pretrain:
            w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
            try: self.net = torchvision.models.resnet18(weights=w)
            except Exception as e:
                if not SMOKE: raise
                print("SMOKE: bobot ImageNet gagal diunduh, pakai bobot acak:", e); self.net = torchvision.models.resnet18(weights=None)
        else: self.net = torchvision.models.resnet18(weights=None)
        self.net.fc = nn.Sequential(nn.Dropout(p_drop), nn.Linear(self.net.fc.in_features, k))
    def forward(self, x):
        if x.shape[1] == 1: x = x.repeat(1, 3, 1, 1)
        if self.uk and x.shape[-1] != self.uk: x = F.interpolate(x, size=(self.uk, self.uk), mode="bilinear", align_corners=False)
        return self.net((x - MEAN_IN.to(x.device)) / SD_IN.to(x.device))

@torch.no_grad()
def logit_semua(model, X, bs=512, praproses=None):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].to(DEV).float().div(255)
        if praproses: x = praproses(x)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(model(x).float())
    return torch.cat(out)

def latih(nama, model, X, Y, Xv, Yv, fungsi_loss, skor_val, maks_epoch=30, patience=5, aug=None, lr=1e-3, bs=128,
          tahap=None, bobot_sampel=None, praproses=None, seed=SEED, wd=1e-4):
    """Latih dengan AMP; early stopping memantau skor_val(logit_val, y_val) (lebih tinggi lebih baik).
    tahap: daftar (n_epoch, lr, beku_backbone). bobot_sampel: oversampling (sampling dengan pengembalian)."""
    seed_semua(seed); model = model.to(DEV); Yd = torch.as_tensor(Y).to(DEV); Yvd = torch.as_tensor(Yv).to(DEV)
    if SMOKE: maks_epoch, patience = min(maks_epoch, 2), 1
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); log = []; best = (-np.inf, None, 0); ep_g = 0
    g = torch.Generator(device="cpu").manual_seed(seed)
    for n_ep, lr_t, beku in (tahap or [(maks_epoch, lr, False)]):
        if SMOKE: n_ep = min(n_ep, 2)
        for nm, p in model.named_parameters(): p.requires_grad = (not beku) or nm.startswith("net.fc")
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_t, weight_decay=wd); tunggu = 0
        for ep in range(n_ep):
            model.train(); t0 = time.perf_counter()
            if bobot_sampel is not None: perm = torch.multinomial(torch.as_tensor(bobot_sampel, dtype=torch.float), len(Y), replacement=True, generator=g)
            else: perm = torch.randperm(len(Y), generator=g)
            perm = perm.to(X.device); tl = 0.0; n = 0
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]; x = X[b].to(DEV).float().div(255)
                if aug: x = augmentasi_gpu(x, **aug)
                if praproses: x = praproses(x)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    lg = model(x)
                loss = fungsi_loss(lg.float(), Yd[b.to(DEV)])
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += loss.item() * len(b); n += len(b)
            if DEV.type == "cuda": torch.cuda.synchronize()
            dt = time.perf_counter() - t0; ep_g += 1
            lv = logit_semua(model, Xv, praproses=praproses); vl = fungsi_loss(lv, Yvd).item(); sv = float(skor_val(lv, Yv))
            log.append({"model": nama, "epoch": ep_g, "tahap_beku": beku, "loss_latih": tl / n, "loss_val": vl, "skor_val": sv, "detik_epoch": dt})
            print(f"{nama} ep{ep_g:02d} loss {tl/n:.4f}/{vl:.4f} skor_val {sv:.4f} {dt:.1f}s")
            if sv > best[0]: best = (sv, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep_g); tunggu = 0
            else:
                tunggu += 1
                if not beku and tunggu >= patience: print(f"  early stopping (patience {patience})"); break
    model.load_state_dict(best[1])
    return model, pd.DataFrame(log), best[2]

def n_param(m): return sum(p.numel() for p in m.parameters())

def ci_bootstrap(fn, y, p, n_boot=1000, seed=SEED, strata=True):
    """CI 95 % persentil; resampling bertingkat per kelas agar setiap sampel memuat semua kelas."""
    r = np.random.default_rng(seed); y = np.asarray(y); idx_k = [np.where(y == k)[0] for k in np.unique(y)]; v = []
    for _ in range(n_boot):
        i = np.concatenate([r.choice(ik, len(ik)) for ik in idx_k]) if strata else r.choice(len(y), len(y))
        try: v.append(fn(y[i], p[i]))
        except ValueError: pass
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def gambar_kurva(LOG, judul_skor, path):
    k = len(LOG); fig, ax = plt.subplots(1, k, figsize=(4.2 * k, 3.2), squeeze=False); ax = ax[0]
    for a, lg in zip(ax, LOG):
        a.plot(lg.epoch, lg.loss_latih, "o-", ms=3, label="loss latih"); a.plot(lg.epoch, lg.loss_val, "s-", ms=3, label="loss val")
        a2 = a.twinx(); a2.plot(lg.epoch, lg.skor_val, "^-", ms=3, color="tab:green", label=judul_skor); a2.set_ylabel(judul_skor, color="tab:green")
        eb = int(lg.loc[lg.skor_val.idxmax(), "epoch"]); a.axvline(eb, color="red", ls="--", lw=1, label=f"terbaik ep {eb}")
        if lg.tahap_beku.any(): a.axvspan(0.5, lg[lg.tahap_beku].epoch.max() + 0.5, color="gray", alpha=0.15, label="tahap beku")
        a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True)); a.set_title(lg.model.iloc[0], fontsize=8); a.set_xlabel("epoch"); a.legend(fontsize=6)
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight"); plt.show()
# ---------- Pemuat HC18 (CC BY 4.0; van den Heuvel dkk., 2018) ----------
from PIL import Image
from scipy import ndimage as ndi
from skimage import measure
def isi_mask(outline):
    m = ndi.binary_fill_holes(outline)
    if m.sum() < outline.sum() * 3:                 # garis elips tidak tertutup rapat -> tutup dulu
        m = ndi.binary_fill_holes(ndi.binary_closing(outline, iterations=3))
    return m
def keliling_elips_mm(mask, pixel_mm):
    """Paskan elips (momen orde-2) pada komponen terbesar, keliling Ramanujan, dikali ukuran piksel (mm)."""
    lab = measure.label(mask)
    if lab.max() == 0: return np.nan
    p = max(measure.regionprops(lab), key=lambda r: r.area); a, b = p.axis_major_length / 2, p.axis_minor_length / 2
    return float(np.pi * (3 * (a + b) - np.sqrt((3 * a + b) * (a + 3 * b))) * pixel_mm)
def muat_hc18(uk, maks=None):
    folder = pastikan_dataset("hc18", ["training_set.zip", "training_set_pixel_size_and_HC.csv"])
    akar = ekstrak_lokal("hc18", "training_set.zip") / "training_set"
    meta = pd.read_csv(folder / "training_set_pixel_size_and_HC.csv"); meta.columns = ["berkas", "pixel_mm", "hc_mm"]
    if maks: meta = meta.iloc[:maks].copy()
    meta["subjek"] = meta.berkas.str.extract(r"^(\d+)")[0]      # 014_HC, 014_2HC, 014_3HC -> subjek 014 (asumsi)
    def baca(fn):
        im = Image.open(akar / fn).convert("L"); w, h = im.size
        m = isi_mask(np.array(Image.open(akar / fn.replace(".png", "_Annotation.png")).convert("L")) > 127)
        return (np.array(im.resize((uk, uk), Image.BILINEAR)), np.array(Image.fromarray(m.astype(np.uint8) * 255).resize((uk, uk), Image.NEAREST)) > 127, w, h)
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(8) as ex: hasil = list(ex.map(baca, meta.berkas))
    X = np.stack([r[0] for r in hasil]); M = np.stack([r[1] for r in hasil]); meta["lebar"] = [r[2] for r in hasil]; meta["tinggi"] = [r[3] for r in hasil]
    return X, M, meta.reset_index(drop=True), akar
def bagi_subjek(meta, seed=42, uji=0.2, val=0.1):
    from sklearn.model_selection import GroupShuffleSplit
    g = meta.subjek.values; i = np.arange(len(meta))
    a, t = next(GroupShuffleSplit(1, test_size=uji, random_state=seed).split(i, groups=g))
    b, v = next(GroupShuffleSplit(1, test_size=val / (1 - uji), random_state=seed).split(a, groups=g[a]))
    return {"train": a[b], "val": a[v], "test": t}
from sklearn.metrics import cohen_kappa_score, confusion_matrix, mean_absolute_error

## 7.1 Data HC18
Citra asli 800×540 piksel diubah ke kotak `UK×UK`. Karena ukuran piksel (mm) berbeda antarcitra, target yang dipelajari model adalah
**HC dalam satuan lebar citra**: `t = HC_mm / (pixel_mm × lebar_px)`; prediksi dikembalikan ke mm dengan mengalikan `pixel_mm × lebar_px`.
Target kemudian distandardisasi (z-score dari set latih).

In [ ]:
UK = 64 if SMOKE else 256
with catat_waktu("2_muat_hc18"):
    XH, MH, META, AKAR = muat_hc18(UK, maks=120 if SMOKE else None)
    SPL = bagi_subjek(META, seed=SEED)
    assert not (set(META.subjek[SPL["train"]]) & set(META.subjek[SPL["test"]])) and not (set(META.subjek[SPL["val"]]) & set(META.subjek[SPL["test"]]))
SKALA = (META.pixel_mm * META.lebar).values                    # mm per "satu lebar citra"
T_REL = META.hc_mm.values / SKALA
MU, SD = T_REL[SPL["train"]].mean(), T_REL[SPL["train"]].std()
Z = ((T_REL - MU) / SD).astype(np.float32)
XHT = ke_tensor_uint8(XH)
T71 = pd.DataFrame([{"split": s, "n_citra": len(i), "n_subjek": META.subjek[i].nunique(), "HC_mm_rerata": META.hc_mm[i].mean(),
                     "HC_mm_sd": META.hc_mm[i].std(), "HC_mm_min": META.hc_mm[i].min(), "HC_mm_maks": META.hc_mm[i].max(),
                     "pixel_mm_min": META.pixel_mm[i].min(), "pixel_mm_maks": META.pixel_mm[i].max()} for s, i in SPL.items()])
T71.to_csv(OUT / "tabel_7_1_data_hc18.csv", index=False)
fig, ax = plt.subplots(2, 5, figsize=(13, 4.6)); r = np.random.default_rng(SEED)
for j, i in enumerate(r.choice(SPL["train"], 5, replace=False)):
    ax[0, j].imshow(XH[i], cmap="gray"); ax[0, j].set_title(f"HC {META.hc_mm[i]:.1f} mm", fontsize=8)
    ax[1, j].imshow(XH[i], cmap="gray"); ax[1, j].contour(MH[i], [0.5], colors="lime", linewidths=1)
    ax[0, j].axis("off"); ax[1, j].axis("off")
ax[1, 0].set_title("anotasi elips (diisi)", fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(OUT / "gambar_7_1_contoh_hc18.png", bbox_inches="tight"); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(META.hc_mm, bins=40); ax[0].set_xlabel("HC (mm)"); ax[0].set_ylabel("jumlah citra")
ax[1].scatter(META.pixel_mm, META.hc_mm, s=4, alpha=0.5); ax[1].set_xlabel("ukuran piksel (mm)"); ax[1].set_ylabel("HC (mm)")
plt.tight_layout(); plt.savefig(OUT / "gambar_7_2_sebaran_hc.png", bbox_inches="tight"); plt.show(); T71

## 7.2 Regresi langsung HC: ResNet-18 pra-latih vs acak (3 seed)
*Loss*: Smooth-L1 pada target z. *Early stopping*: MAE validasi (mm), patience 8. Augmentasi: flip horizontal/vertikal, geser ±5 %,
kecerahan/kontras ±15 % — **tanpa zoom/skala**, karena memperbesar citra akan mengubah HC yang seharusnya diprediksi.

In [ ]:
def ke_mm(z, idx): return (z * SD + MU) * SKALA[idx]
def skor_mae_val(lv, yv): return -float(np.abs(ke_mm(lv[:, 0].cpu().numpy(), SPL["val"]) - META.hc_mm.values[SPL["val"]]).mean())
loss_hc = lambda lg, y: F.smooth_l1_loss(lg[:, 0], y, beta=0.5)
AUG_HC = dict(flip_h=True, flip_v=True, geser=0.05, kecerahan=0.15, kontras=0.15)
SEEDS = [0] if SMOKE else [0, 1, 2]
Xtr, Xv, Xte = XHT[SPL["train"]], XHT[SPL["val"]], XHT[SPL["test"]]
ytr, yv = torch.as_tensor(Z[SPL["train"]]), Z[SPL["val"]]
KONFIG_HC = {"ResNet-18 pra-latih (2 tahap)": (True, [(3, 1e-3, True), (40, 3e-4, False)]),
             "ResNet-18 acak": (False, [(40, 1e-3, False)])}
HASIL_HC, LOG_HC, PRED_HC = [], {}, {}
yte_mm = META.hc_mm.values[SPL["test"]]
for nk, (pre, tahap) in KONFIG_HC.items():
    for s in SEEDS:
        with catat_waktu(f"latih_hc_{'pre' if pre else 'acak'}_s{s}"):
            m, lg, ep = latih(f"{nk} | seed {s}", ResNet18(1, pre), Xtr, ytr, Xv, yv, loss_hc, skor_mae_val, patience=8, aug=AUG_HC, tahap=tahap, bs=32, seed=s)
        p_mm = ke_mm(logit_semua(m, Xte)[:, 0].cpu().numpy(), SPL["test"]); PRED_HC[(nk, s)] = p_mm; LOG_HC[(nk, s)] = lg
        e = p_mm - yte_mm
        HASIL_HC.append({"konfigurasi": nk, "seed": s, "epoch_terbaik": ep, "epoch_dijalankan": int(lg.epoch.max()), "detik_per_epoch": float(lg.detik_epoch.mean()),
                         "MAE_val_mm": -lg.skor_val.max(), "MAE_uji_mm": float(np.abs(e).mean()), "RMSE_uji_mm": float(np.sqrt((e ** 2).mean())),
                         "bias_uji_mm": float(e.mean()), "MAPE_uji_persen": float(np.mean(np.abs(e) / yte_mm) * 100)})
        print(HASIL_HC[-1])
        if pre and s == SEEDS[0]: torch.save(m.state_dict(), OUT / "resnet18_hc18_regresi.pt")
base = META.hc_mm.values[SPL["train"]].mean()
T72 = pd.DataFrame(HASIL_HC); T72.to_csv(OUT / "tabel_7_2_hc_per_seed.csv", index=False)
T73 = T72.groupby("konfigurasi")[["MAE_val_mm", "MAE_uji_mm", "RMSE_uji_mm", "bias_uji_mm", "MAPE_uji_persen", "epoch_terbaik", "detik_per_epoch"]].agg(["mean", "std"])
T73.columns = ["_".join(c) for c in T73.columns]
T73.loc["pembanding: selalu menebak rerata latih", "MAE_uji_mm_mean"] = float(np.abs(base - yte_mm).mean())
T73.loc["pembanding: selalu menebak rerata latih", "RMSE_uji_mm_mean"] = float(np.sqrt(((base - yte_mm) ** 2).mean()))
T73.to_csv(OUT / "tabel_7_3_hc_ringkas.csv"); pd.concat(LOG_HC.values()).to_csv(OUT / "log_epoch_hc.csv", index=False)
gambar_kurva([LOG_HC[(k, SEEDS[0])] for k in KONFIG_HC], "−MAE val (mm)", OUT / "gambar_7_3_learning_curve_hc.png")
T73

## 7.3 Prediksi vs acuan, Bland–Altman, dan galat per rentang HC
Model yang dianalisis: konfigurasi pra-latih dengan **MAE validasi terkecil** (dipilih tanpa melihat set uji).

In [ ]:
kand = [(k, s) for (k, s) in PRED_HC if "pra-latih" in k]
PIL_HC = min(kand, key=lambda ks: T72[(T72.konfigurasi == ks[0]) & (T72.seed == ks[1])].MAE_val_mm.iloc[0]); p_mm = PRED_HC[PIL_HC]
e = p_mm - yte_mm; rata = (p_mm + yte_mm) / 2; bias = e.mean(); loa = (bias - 1.96 * e.std(ddof=1), bias + 1.96 * e.std(ddof=1))
mae_ci = ci_bootstrap(lambda y, p: np.abs(p - y).mean(), yte_mm, p_mm, n_boot=200 if SMOKE else 1000, strata=False)
BA_HC = {"model": f"{PIL_HC[0]} | seed {PIL_HC[1]}", "n_uji": len(e), "MAE_mm": float(np.abs(e).mean()), "MAE_CI95": mae_ci,
         "bias_mm": float(bias), "LoA_bawah_mm": float(loa[0]), "LoA_atas_mm": float(loa[1]),
         "persen_|galat|<=3mm": float((np.abs(e) <= 3).mean() * 100), "persen_|galat|<=5mm": float((np.abs(e) <= 5).mean() * 100)}
print(BA_HC)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].scatter(yte_mm, p_mm, s=8, alpha=0.6); lim = [yte_mm.min() - 10, yte_mm.max() + 10]; ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set_xlabel("HC acuan (mm)"); ax[0].set_ylabel("HC prediksi (mm)"); ax[0].set_title(f"Uji: MAE {BA_HC['MAE_mm']:.2f} mm")
ax[1].scatter(rata, e, s=8, alpha=0.6); ax[1].axhline(bias, color="r", label=f"bias {bias:.2f} mm")
for v in loa: ax[1].axhline(v, color="r", ls="--")
ax[1].set_xlabel("rerata (prediksi, acuan) (mm)"); ax[1].set_ylabel("prediksi − acuan (mm)"); ax[1].set_title(f"Bland–Altman, LoA [{loa[0]:.1f}, {loa[1]:.1f}] mm"); ax[1].legend(fontsize=7)
bins = pd.cut(yte_mm, [0, 100, 150, 200, 250, 400]); g = pd.DataFrame({"rentang": bins, "abs": np.abs(e)}).groupby("rentang", observed=True)["abs"].agg(["mean", "count"])
ax[2].bar(g.index.astype(str), g["mean"]); [ax[2].text(i, v, f"n={c}", ha="center", va="bottom", fontsize=7) for i, (v, c) in enumerate(zip(g["mean"], g["count"]))]
ax[2].set_xlabel("rentang HC acuan (mm)"); ax[2].set_ylabel("MAE (mm)"); ax[2].set_title("Galat per rentang HC")
plt.tight_layout(); plt.savefig(OUT / "gambar_7_4_prediksi_bland_altman.png", bbox_inches="tight"); plt.show()
g.to_csv(OUT / "tabel_7_4_mae_per_rentang.csv")
idx_buruk = np.argsort(-np.abs(e))[:5]
fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
for a, i in zip(ax, idx_buruk):
    a.imshow(XH[SPL["test"][i]], cmap="gray"); a.set_title(f"acuan {yte_mm[i]:.0f} / pred {p_mm[i]:.0f} mm", fontsize=7); a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "gambar_7_5_galat_terbesar.png", bbox_inches="tight"); plt.show()
pd.DataFrame({"berkas": META.berkas.values[SPL["test"]], "hc_acuan_mm": yte_mm, "hc_prediksi_mm": p_mm}).to_csv(OUT / "prediksi_uji_hc.csv", index=False)

## 7.4 RetinaMNIST: derajat retinopati sebagai masalah ordinal
Tiga formulasi dengan ResNet-18 pra-latih (2 tahap) dan augmentasi yang sama:
- **CE**: 5 kelas nominal, prediksi = argmax.
- **Regresi**: 1 keluaran, MSE terhadap derajat 0–4; prediksi = pembulatan (dipotong ke 0–4).
- **Ordinal kumulatif (gaya CORAL)**: 4 keluaran biner `P(y > k)`, k = 0..3; prediksi = jumlah `P > 0,5`.

Metrik utama **QWK** (`cohen_kappa_score(weights="quadratic")`): kesalahan dua derajat dihukum lebih berat daripada satu derajat.
Catatan: set validasi RetinaMNIST hanya 120 citra, sehingga *early stopping* berbasis QWK validasi cukup bising.

In [ ]:
UKR = 28 if SMOKE else 128
with catat_waktu("3_muat_retina"):
    assert INFO["retinamnist"]["license"] == "CC BY 4.0"
    d = npz_medmnist("retinamnist", UKR)
    DR = {s: (d[f"{s}_images"], d[f"{s}_labels"].reshape(-1).astype(np.int64)) for s in ("train", "val", "test")}
    if SMOKE: DR = {s: (x[:200], y[:200]) for s, (x, y) in DR.items()}
XR = {s: ke_tensor_uint8(DR[s][0]) for s in DR}; YR = {s: DR[s][1] for s in DR}
T75 = pd.DataFrame([{"split": s, "n": len(YR[s]), **{f"derajat_{k}": int((YR[s] == k).sum()) for k in range(5)}} for s in DR]); T75.to_csv(OUT / "tabel_7_5_data_retina.csv", index=False)
def pred_ce(lg): return lg.argmax(1).cpu().numpy()
def pred_reg(lg): return np.clip(np.rint(lg[:, 0].cpu().numpy()), 0, 4).astype(int)
def pred_ord(lg): return (torch.sigmoid(lg) > 0.5).sum(1).cpu().numpy()
def loss_ord(lg, y): return F.binary_cross_entropy_with_logits(lg, (y[:, None] > torch.arange(4, device=y.device)[None]).float())
FORMULASI = {"CE (nominal)": (5, lambda lg, y: F.cross_entropy(lg, y), pred_ce),
             "regresi MSE + pembulatan": (1, lambda lg, y: F.mse_loss(lg[:, 0], y.float()), pred_reg),
             "ordinal kumulatif (CORAL)": (4, loss_ord, pred_ord)}
def qwk(y, yh): return cohen_kappa_score(y, yh, weights="quadratic")
AUG_R = dict(flip_h=True, flip_v=True, rot_derajat=15, kecerahan=0.1, kontras=0.1, warna=0.05)
HASIL_R, PRED_R, LOG_R = [], {}, {}
for nf, (k, fl, fp) in FORMULASI.items():
    for s in SEEDS:
        with catat_waktu(f"latih_retina_{nf[:5]}_s{s}"):
            m, lg, ep = latih(f"{nf} | seed {s}", ResNet18(k, True), XR["train"], YR["train"], XR["val"], YR["val"], fl,
                              lambda lv, yv, fp=fp: qwk(yv, fp(lv)), patience=6, aug=AUG_R, tahap=[(2, 1e-3, True), (30, 1e-4, False)], bs=64, seed=s)
        yh = fp(logit_semua(m, XR["test"])); PRED_R[(nf, s)] = yh; LOG_R[(nf, s)] = lg; yt = YR["test"]
        HASIL_R.append({"formulasi": nf, "seed": s, "epoch_terbaik": ep, "QWK_val": lg.skor_val.max(), "QWK_uji": qwk(yt, yh),
                        "akurasi_uji": float((yh == yt).mean()), "MAE_derajat_uji": float(np.abs(yh - yt).mean()),
                        "persen_galat>=2_derajat": float((np.abs(yh - yt) >= 2).mean() * 100)})
        print(HASIL_R[-1])
T76 = pd.DataFrame(HASIL_R); T76.to_csv(OUT / "tabel_7_6_retina_per_seed.csv", index=False)
T77 = T76.groupby("formulasi")[["QWK_val", "QWK_uji", "akurasi_uji", "MAE_derajat_uji", "persen_galat>=2_derajat", "epoch_terbaik"]].agg(["mean", "std"])
T77.columns = ["_".join(c) for c in T77.columns]; T77.to_csv(OUT / "tabel_7_7_retina_ringkas.csv"); pd.concat(LOG_R.values()).to_csv(OUT / "log_epoch_retina.csv", index=False)
T77

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, nf in zip(ax, FORMULASI):
    s = max(SEEDS, key=lambda s: T76[(T76.formulasi == nf) & (T76.seed == s)].QWK_val.iloc[0])
    cm = confusion_matrix(YR["test"], PRED_R[(nf, s)], labels=range(5)); a.imshow(cm, cmap="Blues")
    for i in range(5):
        for j in range(5): a.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
    a.set_xlabel("prediksi"); a.set_ylabel("acuan"); a.set_title(f"{nf}\nseed {s}: QWK uji {qwk(YR['test'], PRED_R[(nf, s)]):.3f}", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "gambar_7_6_konfusi_retina.png", bbox_inches="tight"); plt.show()
# Ilustrasi bobot QWK: penalti kuadratik (i-j)^2/(K-1)^2
W = np.array([[(i - j) ** 2 / 16 for j in range(5)] for i in range(5)]); pd.DataFrame(W).round(3).to_csv(OUT / "tabel_7_8_bobot_qwk.csv")

## 7.5 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(ukuran_hc=UK, ukuran_retina=UKR, data_hc=T71.to_dict("records"), hc_per_seed=T72.to_dict("records"), hc_terpilih=BA_HC, data_retina=T75.to_dict("records"), retina_per_seed=T76.to_dict("records")))))
(OUT / "ringkasan_bab07.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass